# Session 1 — Introduction to Agentic AI

**Salih Tutun, PhD** · DAT 5668 · Module 1 · Concept session (about 80 minutes)

I am Salih Tutun. Gerald Onwujekwe is co-instructor. I talk from the slides first. You answer out loud. Until I say **open the notebook**, this file stays closed.

**Families this session: both.** The agent gets a state, sends an action to the environment, and gets a reward or penalty plus the next state. That is RL. A language model can sit in the same decide step. A DAT 5565 model can help form the state. A public API can sit in as a tool. Same loop. Different machinery.

![The agent loop](../../assets/agent-loop.png)

![One loop, two agent families](../../assets/desk-two-families.png)

## Run of show
- 10 min, slides — What is an agent? Keep this notebook closed until I say open it.
- 22 min, slides — LLM agents, why the firm still needs RL, why most processes need both. Public APIs as tools.
- 33 min, this notebook — Same three refund tickets, LLM-style and RL-style. We type the **Fill this with me** cell together.
- 15 min, slides again — Five questions a manager should ask. Then Session 2.

## Learning goals
After this session students can:
- State the loop as **state → action → environment → reward and next state** (the same as observe → decide → act → feedback).
- Say where a DAT 5565 model sits (a tool that helps form the state) and where an LLM sits (the decide step).
- Say when a step should stay with a person.
- Choose, with a business reason, an LLM-based agent, an RL-based agent, or neither.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I tell you what we are looking at. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor (20 seconds). Two people share. Then type `YOU TYPE HERE`. Then run. |
| **RUN WITH ME** | Already written. Run it. Do not edit. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** out loud — and one line you will keep for your project. |

There is no `TODO` in a concept session. Practice sessions (even numbers) are where you write the function.


## 1. One definition

An agent acts toward a **goal** in an **environment**. It receives a **state**, sends an **action**, and the environment returns a **reward or penalty** and the **next state**. That is RL. We can put an LLM in the decide step of the same loop.

| Step | RL name | Business meaning | Refund desk example |
| --- | --- | --- | --- |
| Observe | state s | Read the current situation | Ticket text, order age, customer tier |
| Decide | policy | Pick the next action | Look up policy, approve, refuse, escalate |
| Act | action a | Change something in the environment | Write a reply or open a case |
| Feedback | reward r, next state s' | Learn whether the action helped | Customer accepted, policy violation, cost |

You already know how to fit a sklearn or Keras model. Here that model can help build s (a score, a forecast). It does not replace the loop. A chatbot that only writes a paragraph never sends a to the environment and never gets r. It is not an agent. An agent is defined by the action and the feedback, not by the model brand.

**Wrong family, wrong job**
- A chat reply is a poor inventory controller: the reward is a number that repeats every day, and the state is stock, not a paragraph.
- A DQN is a poor contract reviewer: the "state" is unstructured language, and the action is a tool call plus a cited reason.


## 2. Worked case — damaged-item refund

Instructional example, not firm data. Policy the business will stand behind:

- Damaged items may be refunded within 30 days.
- Orders older than 30 days are refused unless a manager escalates.
- Refund amount is the order total, not a number the model invents.

We will run **the same three tickets** through both families.


## Optional model, and public APIs as tools

The decide step is a **local stand-in**. It must call `lookup_policy`, then `lookup_amount`, then finish from those tool results. It does not invent a dollar amount.

[public-apis](https://github.com/public-apis/public-apis) lists **Google Gemini** under Machine Learning. Auth is `apiKey` — the list does **not** publish a key. The docs URL is [ai.google.dev/gemini-api/docs](https://ai.google.dev/gemini-api/docs). WashU Google accounts are often blocked at [AI Studio](https://aistudio.google.com/apikey) (age / Workspace). Leave the key empty and the stand-in runs. A personal Gmail (18+) can create a key later; do not put it in a committed file.

Other rows on that list (prefer **No** Auth) are **tools** inside the loop, not the agent: [Frankfurter](https://www.frankfurter.app/docs) for FX is one example.

Calling an API does not make you an agent. Sending action $a$ into that API and taking reward $r$ and next state $s'$ does.


**This cell — public-apis rows we use.** Filled from the catalog. Gemini is decide (Agent box) and still needs a key you already have in the environment. Frankfurter is a No-Auth tool. There is no secret in this cell.


In [ ]:
# RUN WITH ME — copied from https://github.com/public-apis/public-apis
# Machine Learning: Google Gemini (apiKey). Currency Exchange: Frankfurter (No).
PUBLIC_APIS = {
    "google_gemini": {
        "docs": "https://ai.google.dev/gemini-api/docs",
        "auth": "apiKey",
        "role": "decide — sits in the Agent box",
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent",
    },
    "frankfurter": {
        "docs": "https://www.frankfurter.app/docs",
        "auth": "No",
        "role": "tool — environment lookup, not the agent",
        "endpoint": "https://api.frankfurter.dev/v1/latest",
    },
}
PUBLIC_APIS


**This cell — Gemini key.** public-apis does not include the secret. Leave `""` for class (WashU accounts hit the AI Studio block). If `GEMINI_API_KEY` is already set in the OS or in Colab Secrets, this cell will use it. Never print the key.


In [ ]:
# Optional Gemini key. public-apis lists Google Gemini as apiKey — no key is in that list.
# Docs: https://ai.google.dev/gemini-api/docs
# WashU Google → AI Studio is often blocked. Leave "" and the stand-in runs.
import os

GEMINI_API_KEY_FOR_CLASS = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or ""

if GEMINI_API_KEY_FOR_CLASS:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY_FOR_CLASS
try:
    from google.colab import userdata
    _colab_key = userdata.get("GEMINI_API_KEY")
    if _colab_key:
        os.environ["GEMINI_API_KEY"] = _colab_key
except Exception:
    pass

_key_on = bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))
print("Gemini key loaded from the environment." if _key_on else "No Gemini key (expected on WashU Google). Decide step: local stand-in.")
print("Catalog: https://ai.google.dev/gemini-api/docs | auth: apiKey")


**This cell — imports.** Standard library only. `json` and `urllib` are for the Gemini HTTP call. No sklearn here. Run it and keep going.


In [ ]:
# RUN WITH ME
import json
import os
import urllib.request


**This cell — tickets = state $s$.** Both families see the same three observations. A is cracked and inside 30 days. B is not damage. C is cracked and stale. Guess the actions out loud before we code: refund, refuse, or escalate.


In [ ]:
# RUN WITH ME — shared observations
TICKETS = [
    {"id": "A", "text": "Item arrived cracked", "order_age_days": 12, "order_total": 80},
    {"id": "B", "text": "Changed my mind", "order_age_days": 10, "order_total": 40},
    {"id": "C", "text": "Screen is cracked", "order_age_days": 45, "order_total": 200},
]
TICKETS


**This cell — the policy tool's table.** This is the rule the business will stand behind. Damaged items: refund if age ≤ 30. Changed mind: never refund. The model is not allowed to invent a fourth rule.


In [ ]:
# RUN WITH ME — the model does not invent the policy
POLICY = {
    "damaged": {"max_age_days": 30, "outcome": "refund"},
    "changed mind": {"max_age_days": 0, "outcome": "refuse"},
}
POLICY


**This cell — `lookup_policy`.** Tool 1. It reads the ticket text and returns a row from `POLICY`, or `None`. Point at A → damaged. B → changed mind. C → damaged (age is checked later, not here).


In [ ]:
# RUN WITH ME — tool 1
def lookup_policy(text):
    """Observe the rule that applies. This is a tool, not a guess."""
    lowered = text.lower()
    if "crack" in lowered:
        return POLICY["damaged"]
    if "changed my mind" in lowered:
        return POLICY["changed mind"]
    return None


print("A:", lookup_policy(TICKETS[0]["text"]))
print("B:", lookup_policy(TICKETS[1]["text"]))
print("C:", lookup_policy(TICKETS[2]["text"]))


**This cell — tool schema.** These are the only names the LLM-based agent may choose. `finish` is how it sends action $a$. A chatbot has no schema, so it never acts.


In [ ]:
# RUN WITH ME — schema. The agent cannot invent a fourth tool.
TOOL_SCHEMA = {
    "lookup_policy": "Read the refund rule for this ticket text.",
    "lookup_amount": "Read the order total from the system of record.",
    "finish": "Stop and return refund, refuse, or escalate.",
}
list(TOOL_SCHEMA)


**This cell — `run_tool`.** Tool 2 is `lookup_amount`: the dollar figure is `order_total` from the system of record. Never 999. Watch the printout for ticket A: it must be 80.


In [ ]:
# RUN WITH ME — amount comes from the order, never from the model
def run_tool(tool, ticket):
    if tool == "lookup_policy":
        return lookup_policy(ticket["text"])
    if tool == "lookup_amount":
        return ticket["order_total"]
    raise ValueError(tool)


print("Amount on A:", run_tool("lookup_amount", TICKETS[0]))


**This cell — `local_decide`.** This is the Agent box without Gemini. Same contract as a tool-calling model: name the next tool, then `finish` with refund / refuse / escalate. First call on A must be `lookup_policy`. It does not invent a dollar amount.


In [ ]:
# RUN WITH ME — same contract as a tool-calling model: name the next tool, not a dollar amount
def local_decide(ticket, memory):
    called = [step["tool"] for step in memory]
    if "lookup_policy" not in called:
        return {"tool": "lookup_policy"}
    if "lookup_amount" not in called:
        return {"tool": "lookup_amount"}
    rule = next(step["result"] for step in memory if step["tool"] == "lookup_policy")
    if rule is None:
        action, reason = "escalate", "No policy tool matched the ticket."
    elif ticket["order_age_days"] <= rule["max_age_days"] and rule["outcome"] == "refund":
        action, reason = "refund", f"Policy allows a refund within {rule['max_age_days']} days."
    else:
        action, reason = "refuse", "Policy tool does not allow this refund."
    return {"tool": "finish", "action": action, "reason": reason}


print("First call on A:", local_decide(TICKETS[0], []))


**This cell — `gemini_decide`.** Gemini sits in the Agent box. It may only pick a name from `TOOL_SCHEMA`. No key, a refused call, or a bad JSON → `None`, and we fall back. If you see `None` on A, that is expected without a key.


In [ ]:
# RUN WITH ME — returns None when no key is set or the call is unusable
def gemini_decide(ticket, memory):
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not key:
        return None
    model = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
    prompt = {
        "instruction": (
            "You are the decide step of a tool-using agent. "
            "Reply with JSON only: {\"tool\": <one name from schema>, "
            "\"action\": <refund|refuse|escalate, only if tool is finish>, "
            "\"reason\": <one sentence from the tool results>} . "
            "Do not invent a refund amount. Do not name a tool that is not in the schema."
        ),
        "schema": TOOL_SCHEMA,
        "ticket": {"id": ticket["id"], "text": ticket["text"], "order_age_days": ticket["order_age_days"]},
        "memory": memory,
    }
    body = json.dumps({
        "contents": [{"role": "user", "parts": [{"text": json.dumps(prompt)}]}],
        "generationConfig": {"temperature": 0, "responseMimeType": "application/json"},
    }).encode()
    endpoint_tmpl = (
        PUBLIC_APIS["google_gemini"]["endpoint"]
        if "PUBLIC_APIS" in globals()
        else "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    )
    request = urllib.request.Request(
        endpoint_tmpl.format(model=model),
        data=body,
        headers={"Content-Type": "application/json", "x-goog-api-key": key},
    )
    try:
        with urllib.request.urlopen(request, timeout=12) as response:
            payload = json.loads(response.read().decode())
        text = payload["candidates"][0]["content"]["parts"][0]["text"]
        decision = json.loads(text)
    except Exception:
        return None
    tool = decision.get("tool") if isinstance(decision, dict) else None
    if tool not in TOOL_SCHEMA:
        return None
    if tool != "finish":
        return {"tool": tool}
    action = decision.get("action")
    if action not in ("refund", "refuse", "escalate"):
        return None
    reason = decision.get("reason") or "Finished from the tool results."
    return {"tool": "finish", "action": action, "reason": reason}


print("Gemini first call on A:", gemini_decide(TICKETS[0], []))


**This cell — `llm_decide`.** One decide function for the LLM family. Try Gemini; if it returns `None`, use the stand-in. The printout tells you which source ran. The boxes do not change.


In [ ]:
# RUN WITH ME — Gemini when the reply is valid; otherwise the local stand-in
def llm_decide(ticket, memory):
    remote = gemini_decide(ticket, memory)
    if remote is not None:
        llm_decide.source = "gemini"
        return remote
    llm_decide.source = "local stand-in"
    return local_decide(ticket, memory)


print("Decide on A:", llm_decide(TICKETS[0], []), "| source:", getattr(llm_decide, "source", "?"))


**This cell — `llm_style_agent`.** This is the loop: observe the ticket, decide (tool or finish), act through `run_tool`, store the result, repeat. `finish` is action $a$. The amount on a refund is the order total. That is an LLM-based agent. A paragraph-only chatbot never enters this cell.


In [ ]:
# RUN WITH ME — the loop. Language in. Booked decision out.
def llm_style_agent(ticket):
    memory = []
    for _ in range(4):
        decision = llm_decide(ticket, memory)
        if decision["tool"] == "finish":
            amount = ticket["order_total"] if decision["action"] == "refund" else 0
            return {
                "action": decision["action"],
                "reason": decision["reason"],
                "steps": [step["tool"] for step in memory],
                "source": getattr(llm_decide, "source", "?"),
                "feedback": {"auditable": True, "amount": amount},
            }
        memory.append({"tool": decision["tool"], "result": run_tool(decision["tool"], ticket)})
    return {"action": "escalate", "reason": "Agent exceeded its step budget.", "steps": [], "source": "?", "feedback": {"auditable": True, "amount": 0}}


**This cell — run A, B, C with the LLM family.** Say the three actions before you run. You should see A refund \$80, B refuse \$0, C refuse \$0. Both tools in `steps`. If Gemini is on, `source` says gemini; otherwise local stand-in.


In [ ]:
# RUN WITH ME — same three tickets, LLM family
_key_set = bool(os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"))
print("Decide step:", "Gemini if the call succeeds, else stand-in" if _key_set else "local stand-in")
for ticket in TICKETS:
    result = llm_style_agent(ticket)
    print(
        f"Ticket {ticket['id']}: steps={result['steps']} -> {result['action']:8} | "
        f"{result['reason']} | amount={result['feedback']['amount']} | {result['source']}"
    )


## 3. The same loop as an RL-based agent

An RL-based agent does not read the sentence and does not call those tools. Someone has already mapped the ticket to a **state**. The agent picks an **action** because that action has the best learned value. Feedback is a **reward**, not a citation. Sessions 4 through 10 train this family. Sessions 1 and 3 train the LLM-based family. Sessions 11 and 12 combine them.

State used here (instructional, deliberately small):

- `fresh_damage` — damaged and inside the window
- `stale_damage` — damaged and outside the window
- `other` — anything else

Actions: `refund`, `refuse`, `escalate`.


**This cell — Q-table.** Session 8 trains this. Today we only read it. Each number is expected long-run reward, not a probability. Highest value in the row is the action. `other` prefers escalate (2.5) over refuse (2.0). That is why ticket B will disagree with the LLM family.


In [ ]:
# RUN WITH ME — values are expected long-run reward, not probabilities
Q = {
    "fresh_damage": {"refund": 8.0, "refuse": -2.0, "escalate": 1.0},
    "stale_damage": {"refund": -6.0, "refuse": 3.0, "escalate": 2.0},
    "other": {"refund": -4.0, "refuse": 2.0, "escalate": 2.5},
}
Q


**This cell — `to_state`.** The RL agent does not read the sentence. Someone already mapped text + age to three states. A → `fresh_damage`. B → `other`. C → `stale_damage`. If this mapping is wrong, the learned policy is wrong. That is a design fact.


In [ ]:
# RUN WITH ME — map the ticket to a small state. The RL agent does not read the sentence.
def to_state(ticket):
    damaged = "crack" in ticket["text"].lower()
    if damaged and ticket["order_age_days"] <= 30:
        return "fresh_damage"
    if damaged:
        return "stale_damage"
    return "other"


for ticket in TICKETS:
    print(f"Ticket {ticket['id']}: {to_state(ticket)}")


**This cell — `rl_style_agent`.** Observe `to_state`, decide greedy on $Q$, treat the Q-value as the reward signal we show today. No tools. No citation. Same four boxes as Sutton. Different machinery inside the Agent box.


In [ ]:
# RUN WITH ME — greedy on the learned values
def rl_style_agent(ticket):
    state = to_state(ticket)
    action = max(Q[state], key=Q[state].get)
    reward = Q[state][action]
    return {"observation": state, "action": action, "value": reward}


**This cell — run A, B, C with the RL family.** Same tickets. You should see A refund, B escalate, C refuse. Ticket B is the disagreement: LLM refused from the policy text; RL escalates `other`. A person should see that.


In [ ]:
# RUN WITH ME — same three tickets, RL family
print("RL-style policy on the same tickets")
for ticket in TICKETS:
    result = rl_style_agent(ticket)
    print(f"Ticket {ticket['id']}: state={result['observation']:14} action={result['action']:8} value={result['value']}")


## 4. What to say to a manager

Ask these five questions before anyone builds the agent. They are the same questions for both families.

1. **Goal** — What decision, if automated, would change a KPI this quarter?
2. **Data** — Is the input a document/ticket, or a repeated numeric state?
3. **Actions** — What may the system do, and what is forbidden?
4. **Feedback** — Is the signal a citation and a human rating, or a numeric reward every step?
5. **Human checkpoint** — Which action is too costly to take without a person?

| Ticket | LLM-style action | RL-style action | Checkpoint |
| --- | --- | --- | --- |
| A, cracked, day 12 | refund, with the policy tool | refund | Auto-approve inside policy |
| B, changed mind | refuse | escalate (`other` has no damage rule) | A person should see "other" |
| C, cracked, day 45 | refuse | refuse | Do not let either family override the window |

The families can disagree (ticket B). That disagreement is a design fact, not a bug. Session 2. You will build the tool-using side and see what fails if the tool is skipped.

## Bridge to Session 2
Students will replace a reply-only chatbot with an agent that must call `lookup_policy`, `lookup_order`, and `open_case`.


## Fill this with me — type this in class

**Stop.** Do not run the next two cells until we type them together. Guess out loud first.

1. Type `guess_llm` and `guess_rl` as `refund`, `refuse`, or `escalate` for a cracked lid on day 20. Run that cell.
2. Run the ticket cell. Then set `order_age_days` to `60` and run again.
3. Then set `text` to `"Changed my mind"` and run again. Watch the families disagree.


**This cell — your guesses.** Cracked lid, day 20: both families should refund. Type `refund` in both blanks. Say it out loud before you run.


In [ ]:
# FILL WITH THE CLASS. Type before you run. Allowed: "refund", "refuse", "escalate".
guess_llm = ""  # <-- type here
guess_rl = ""   # <-- type here
print("You said LLM:", guess_llm or "(type it first)", "| RL:", guess_rl or "(type it first)")


**This cell — ticket D, both families.** Run once: both refund. Change `order_age_days` to `60`: both refuse (outside the window). Change `text` to `"Changed my mind"`: LLM refuses from the policy tool; RL escalates `other`. That disagreement is the hybrid: a person still owns the exception.


In [ ]:
# FILL WITH THE CLASS — then edit this ticket and run again
ticket_d = {"id": "D", "text": "Lid arrived cracked", "order_age_days": 20, "order_total": 50}
llm = llm_style_agent(ticket_d)
rl = rl_style_agent(ticket_d)
print("LLM-based agent:", llm["action"], "| you said:", guess_llm or "(type it first)")
print("RL-based agent: ", rl["action"], "| you said:", guess_rl or "(type it first)")
print("Now set order_age_days to 60, run again. Then change text to 'Changed my mind'.")
